# Silverwing-120M — Kaggle GPU Pretraining (Phase B GPU)

Reproducible GPU pretrain of the **120,087,552-parameter** Silverwing-120M-V1 architecture on a Kaggle GPU (T4 / P100 / L4).

Fixes the three Colab-102M blockers:
1. **Data-not-found** → integrated controlled data pipeline builds `train.npy`/`val.npy`.
2. **GPU→CPU fallback** → hard `STOP` if the kernel is not GPU-backed.
3. **Fake resume** → real model(safetensors-strict) + optimizer + scheduler + step restore.

> ⚠️ This authoring host is CPU-only/offline. Cells needing GPU or network are **not** executed here — they are syntax-checked, and the offline-runnable data pipeline is validated against the local Silverwing corpus. The 12M/r2000 baseline is **immutable**.


## 1. Environment audit

In [ ]:
import torch, sys, platform, shutil, os
gpu = torch.cuda.is_available()
print("GPU:",        torch.cuda.get_device_name(0) if gpu else "None (CPU-only)")
print("GPU count:",   torch.cuda.device_count())
if gpu:
    print("VRAM:",       f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
    print("CUDA:",       torch.version.cuda)
    print("BF16:",       torch.cuda.is_bf16_supported())
else:
    print("VRAM: N/A"); print("CUDA: not built"); print("BF16: N/A")
print("PyTorch:",  torch.__version__)
print("Python:",   platform.python_version())
try:
    import psutil
    print("RAM:", f"{psutil.virtual_memory().total/1e9:.2f} GB")
except Exception:
    print("RAM: (psutil unavailable)")
print("Disk:", f"{shutil.disk_usage('/kaggle/working').free/1e9:.1f} GB free")
print("FP16:",  bool(gpu))


## 2. GPU requirement gate — STOP if not GPU-backed

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("STOP: kernel is not GPU-backed. Select a GPU accelerator (T4/P100/L4) and rerun.")
print("GPU-backed accelerator confirmed:", torch.cuda.get_device_name(0))


## 3. Dependency verification

Upload `silverwing_train.py`, `silverwing_120m_config.json`, `stage3a_evaluate.py`, `gen_samples_120m.py`, `silverwing_120m_kaggle.py`, and the r2000 `tokenizer.json` (→ `r2000_tokenizer.json`) to `/kaggle/working`, or attach them as a dataset.

In [ ]:
import sys, os
sys.path.insert(0, "/kaggle/working")
import torch, numpy as np
from safetensors import __version__ as sf_ver
import silverwing_train as sw
print("deps OK | torch", torch.__version__, "| safetensors", sf_ver, "| numpy", np.__version__)
print("sw module:", sw.__file__)
for f in ["silverwing_120m_config.json","r2000_tokenizer.json","silverwing_120m_kaggle.py","stage3a_evaluate.py"]:
    print(f"  {f}:", "present" if os.path.exists('/kaggle/working/'+f) else "MISSING")


## 4. Architecture lock — do NOT rewrite the model

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
import silverwing_train as sw, torch
CFG = Path("/kaggle/working/silverwing_120m_kaggle.py").parent / "silverwing_120m_config.json"
spec = json.load(open(CFG))
cfg = sw.ModelConfig(**spec)
m = sw.SilverwingModel(cfg)
n = m.num_parameters()
EXPECTED = 120087552
print("Parameters:", n, "->", "PASS" if n == EXPECTED else f"FAIL (expected {EXPECTED})")
print("Layers:",        spec.get("n_layers"),       "d_model:",        spec.get("d_model"))
print("Q heads:",       spec.get("n_heads"),        "KV heads:",       spec.get("n_kv_heads"))
print("head_dim:",      spec.get("d_model") // spec.get("n_heads"), "MLP:", spec.get("hidden_size"))
print("RoPE theta:",    spec.get("rope_theta"),     "vocab:",          spec.get("vocab_size"))
print("context:",       spec.get("max_position_embeddings"),
      "weight_tying:", spec.get("tie_word_embeddings"),
      "use_bias:",     spec.get("use_bias"))
print("Model class:", type(m).__name__)
assert n == EXPECTED, "ARCH LOCK FAILED: parameter count != 120,087,552 — STOP"
print("ARCH LOCK: PASS")


## 5. Tokenizer lock — reuse the r2000 8192-token BPE (not rebuilt, not remapped)

In [ ]:
import hashlib, sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
import silverwing_train as sw
TOK = Path("/kaggle/working/r2000_tokenizer.json")
tok = sw.BPETokenizer.from_file(str(TOK))
data = TOK.read_bytes()
print("tokenizer path:", TOK)
print("SHA256:", hashlib.sha256(data).hexdigest())
print("vocabulary size:", tok.vocab_size)
print("special token IDs: pad=0 bos=1 eos=2 unk=3")
assert tok.vocab_size == 8192, "tokenizer vocab != 8192 — STOP"
print("TOKENIZER LOCK: PASS")


## 6. Data mixture + controlled pipeline (experimental, versioned)

In [ ]:

import hashlib, json, os, re, unicodedata, datetime
from pathlib import Path
import numpy as np
import importlib.util
_TRAINER = os.environ.get("SW_TRAINER_PATH", "/kaggle/working/silverwing_120m_kaggle.py")
_spec = importlib.util.spec_from_file_location("kg", _TRAINER)
kg = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(kg)

DATA_MIXTURE = {
    "version": "silverwing-120m-mixture-v0.1",
    "sources": {
        "general":     {"frac": 0.40, "license": "CC-BY-SA"},
        "stem_edu":    {"frac": 0.20, "license": "CC-BY-SA / ODbL"},
        "programming": {"frac": 0.15, "license": "MIT / CC-BY-SA"},
        "science_eng": {"frac": 0.10, "license": "CC-BY-SA"},
        "math_reason": {"frac": 0.10, "license": "CC-BY-SA"},
        "instruction": {"frac": 0.05, "license": "CC-BY-SA"},
    },
    "note": "experimental mixture; not claimed optimal.",
}


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def pipeline_build(raw_text, seq, target_tokens, val_frac, out_dir, tok):
    """Controlled Silverwing pipeline: clean -> ngram-dedup -> tokenize -> pack@seq.
    Returns (train_tokens, val_tokens, doc_count, raw_chars, kept_chars)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    lines = []
    for ln in raw_text.splitlines():
        ln = unicodedata.normalize("NFC", ln).strip()
        if kg.is_english(ln) and kg.quality_filter(ln):
            lines.append(kg.pii_redact(ln))
    raw_chars = len(raw_text)
    lines = kg.ngram_dedup(lines)
    kept_chars = sum(len(l) for l in lines)
    ids = tok.encode("\n".join(lines))
    ids = np.asarray(ids, dtype=np.int32)
    n = (len(ids) // seq) * seq
    ids = ids[:n]
    val_n = max(int(len(ids) * val_frac) // seq * seq, seq * 2)
    val_ids, tr_ids = ids[:val_n], ids[val_n:]
    take = (target_tokens // seq) * seq
    tr_ids = tr_ids[:take]
    tr = tr_ids.reshape(-1, seq); va = val_ids.reshape(-1, seq)
    np.save(out_dir / "train.npy", tr); np.save(out_dir / "val.npy", va)
    return int(len(tr) * seq), int(len(va) * seq), len(lines), raw_chars, kept_chars


def fetch_source_text(corpus_path=None):
    """Controlled Silverwing source: local attached corpus preferred, else a
    CC-BY-SA download (WikiText-103). wikitext is the *source text* only - it is
    never used raw; it always passes through pipeline_build."""
    corpus_path = Path(corpus_path) if corpus_path else Path(os.environ.get("SILWING_CORPUS_TXT", "/kaggle/input/silverwing-corpus/train.txt"))
    if corpus_path.exists():
        return corpus_path.read_text(encoding="utf-8", errors="replace"), str(corpus_path), "local-attached"
    import urllib.request
    urls = [
        "https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-103-raw-v1.zip",
        "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt",
    ]
    for u in urls:
        try:
            txt = urllib.request.urlopen(u, timeout=120).read().decode("utf-8", "replace")
            if len(txt) > 1_000_000:
                return txt, u, "downloaded"
        except Exception as e:
            print("fetch failed:", u, repr(e))
    raise RuntimeError("No corpus available: attach a dataset or enable network.")


def write_dataset_manifest(out_dir, src, src_type, train_tokens, val_tokens,
                           doc_count, raw_chars, kept_chars, tok_path, seq, target_tokens):
    manifest = {
        "source": src,
        "source_type": src_type,
        "version": DATA_MIXTURE["version"],
        "license": "CC-BY-SA",
        "sequence_length": seq,
        "target_tokens": target_tokens,
        "document_count": doc_count,
        "raw_size": raw_chars,
        "filtered_size": kept_chars,
        "token_count": train_tokens + val_tokens,
        "train_tokens": train_tokens,
        "validation_tokens": val_tokens,
        "val_frac": 0.005,
        "tokenizer_path": str(tok_path),
        "tokenizer_hash": sha256_file(tok_path) if Path(tok_path).exists() else "n/a",
        "dataset_hash": hashlib.sha256((src + str(train_tokens) + str(val_tokens)).encode()).hexdigest(),
        "creation_date": datetime.datetime.utcnow().isoformat() + "Z",
        "pipeline": "silverwing clean+ngram_dedup+pack",
        "data_mixture": DATA_MIXTURE,
    }
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    (Path(out_dir) / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2))
    return manifest


## 7. Build a controlled ~5M-token shard (then expand 25M→50M)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working")
import silverwing_train as sw
DATA_ROOT = Path("/kaggle/working/silverwing-120m/data"); DATA_ROOT.mkdir(parents=True, exist_ok=True)
TOK = Path("/kaggle/working/r2000_tokenizer.json")
print("DATA MIXTURE (experimental):")
for k, v in DATA_MIXTURE["sources"].items():
    print(f"  {k:12s} {int(v['frac']*100):>3d}%  ({v['license']})")
print("version:", DATA_MIXTURE["version"])
seq = 2048
target_tokens = 5_000_000
raw, src, src_type = fetch_source_text(TOK)
tok = sw.BPETokenizer.from_file(str(TOK))
tr, va, docs, rc, kc = pipeline_build(raw, seq, target_tokens, 0.005, DATA_ROOT, tok)
man = write_dataset_manifest(DATA_ROOT, src, src_type, tr, va, docs, rc, kc, TOK, seq, target_tokens)
print(f"built shard: train={tr} tokens ({tr/1e6:.2f}M), val={va} tokens, docs={docs}")
print("dedup ratio:", f"{(rc-kc)/rc:.1%}" if rc else "n/a")


## 8. Training configuration (GPU-aware)
T4: FP16 (BF16 unsupported on Turing). Effective batch = `batch*grad_accum`.

In [ ]:
CFG_RUN = dict(
    model="Silverwing-120M-V1", steps=500, seq=2048, batch=1, grad_accum=4,
    lr=1.5e-4, warmup=250, weight_decay=0.01, clip=1.0,
    save_every=100, eval_every=100, amp="fp16", device="cuda", seed=42,
)
print("training config:"); print(CFG_RUN)
print("effective batch =", CFG_RUN["batch"]*CFG_RUN["grad_accum"])
print("OOM fallback: batch1 -> +grad_accum -> grad-checkpoint -> shrink seq (never shrink the model)")


## 9. Throughput benchmark (VRAM fit + step-time estimate)

In [ ]:
import sys, time, json, torch
sys.path.insert(0, "/kaggle/working")
import silverwing_train as sw
cfg = sw.ModelConfig(**json.load(open("/kaggle/working/silverwing_120m_config.json")))
m = sw.SilverwingModel(cfg).to("cuda").train()
opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()
seq=1024; ga=4; N=20
ids = torch.randint(0, 8192, (1, seq), device="cuda")
with torch.cuda.amp.autocast():
    l = m(ids, labels=ids)["loss"]/ga
scaler.scale(l).backward(); scaler.unscale_(opt); scaler.step(opt); scaler.update(); opt.zero_grad()
torch.cuda.synchronize()
t0=time.time()
for _ in range(N):
    opt.zero_grad()
    for _ in range(ga):
        with torch.cuda.amp.autocast():
            l = m(ids, labels=ids)["loss"]/ga
        scaler.scale(l).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
    scaler.step(opt); scaler.update()
torch.cuda.synchronize()
dt=(time.time()-t0)/N; tps=seq*ga
print(f"step_time   = {dt*1000:.1f} ms")
print(f"tokens/sec  = {tps/dt:.0f}")
print(f"VRAM alloc  = {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM resid  = {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"est 500     = {500*dt/60:.1f} min")
print(f"est 5K      = {5000*dt/60:.1f} min")
print(f"est 10K     = {10000*dt/60:.1f} min")


## 10. GPU smoke test — all fixes, must PASS before the long run

In [ ]:
import sys, time, math, tempfile, os, json, torch
sys.path.insert(0, "/kaggle/working")
import silverwing_train as sw
from safetensors.torch import save_file, load_file

cfg = sw.ModelConfig(**json.load(open("/kaggle/working/silverwing_120m_config.json")))
m  = sw.SilverwingModel(cfg).to("cuda"); m.train()
opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()
seq=128; ga=4
ids = torch.randint(0, 8192, (1, seq), device="cuda")

with torch.cuda.amp.autocast():
    out = m(ids, labels=ids)
fwd_ok = torch.isfinite(out["logits"]).all().item()
f_loss = float(out["loss"])

opt.zero_grad()
for _ in range(ga):
    with torch.cuda.amp.autocast():
        l = m(ids, labels=ids)["loss"]/ga
    scaler.scale(l).backward()
scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
scaler.step(opt); scaler.update()
bwd_ok = True

d = tempfile.mkdtemp()
save_file({k:v.clone() for k,v in m.state_dict().items()}, os.path.join(d,"model.safetensors"))
torch.save({"step":5,"opt":opt.state_dict(),"sched":torch.optim.lr_scheduler.LambdaLR(opt,lambda s:1.0).state_dict()}, os.path.join(d,"optimizer.pt"))
m2 = sw.SilverwingModel(cfg).to("cuda")
missing, unexpected = m2.load_state_dict(load_file(os.path.join(d,"model.safetensors")), strict=True)
resume_ok = (len(missing)==0 and len(unexpected)==0 and torch.allclose(next(m.parameters()), next(m2.parameters())))

losses=[f_loss]
for _ in range(5):
    opt.zero_grad(); tot=0.0
    for _ in range(ga):
        with torch.cuda.amp.autocast():
            l = m(ids, labels=ids)["loss"]/ga
        scaler.scale(l).backward(); tot+=l.item()*ga
    scaler.unscale_(opt); scaler.step(opt); scaler.update(); losses.append(tot/ga)
nan_ok = all(math.isfinite(x) for x in losses)
inf_ok = all(not math.isinf(x) for x in losses)

results = {"CUDA":torch.cuda.is_available(),"Forward":fwd_ok,"Backward":bwd_ok,"AMP":bool(scaler.is_enabled()),
           "Optimizer":True,"Checkpoint":os.path.exists(os.path.join(d,"model.safetensors")),
           "Resume":resume_ok,"NaN":nan_ok,"Inf":inf_ok}
print("SILVERWING-120M GPU SMOKE")
for k,v in results.items(): print(f"{k:11s}: {'PASS' if v else 'FAIL'}")
print("STATUS:", "PASS" if all(results.values()) else "FAIL")
assert all(results.values()), "GPU SMOKE FAILED — STOP before real pretrain"


## 11. Launch the 500-step GPU pilot (validated trainer)

The script auto-detects CUDA + AMP, writes `checkpoints/step-N/best/final` each with `model.safetensors`+`optimizer.pt`+`metadata.json`, and degrades to CPU if no GPU (never silent on Kaggle — cell 2 guards that).

In [ ]:
import os, shutil
os.makedirs("/kaggle/working/silverwing-120m", exist_ok=True)
if not os.path.exists("/kaggle/working/r2000_tokenizer.json") and os.path.exists("/kaggle/input/r2000-tokenizer/tokenizer.json"):
    shutil.copy2("/kaggle/input/r2000-tokenizer/tokenizer.json", "/kaggle/working/r2000_tokenizer.json")
    print("copied r2000 tokenizer -> /kaggle/working/r2000_tokenizer.json")
else:
    print("tokenizer present:", os.path.exists("/kaggle/working/r2000_tokenizer.json"))


In [ ]:
%%bash
SW_TOK=/kaggle/working/r2000_tokenizer.json python /kaggle/working/silverwing_120m_kaggle.py \
  --steps 500 --seq 2048 --batch 1 --grad-accum 4 \
  --lr 1.5e-4 --warmup 250 --weight-decay 0.01 \
  --save-every 100 --eval-every 100 \
  --ckpt-dir /kaggle/working/silverwing-120m \
  --data /kaggle/working/silverwing-120m/data/train.npy \
  --val  /kaggle/working/silverwing-120m/data/val.npy

## 12. Checkpoint inventory

In [ ]:
import json, os
from pathlib import Path
root = Path("/kaggle/working/silverwing-120m")
print("checkpoints:")
for p in sorted((root/"checkpoints").glob("step-*")):
    md = json.load(open(p/"metadata.json"))
    print(f"  {p.name}: step={md['step']} sha={md.get('model_sha256','?')[:12]} params={md['param_count']} amp={md.get('amp')}")
for d in ("best","final"):
    f = root/d/"metadata.json"
    if f.exists():
        md = json.load(open(f))
        print(f"  {d}: step={md['step']} sha={md.get('model_sha256','?')[:12]} val_loss={md.get('val_loss')}")
print("final model size:", round(os.path.getsize(root/"final/model.safetensors")/1e6,1), "MB")


## 13. Resume test (real resume -> step 501)

In [ ]:
%%bash
SW_TOK=/kaggle/working/r2000_tokenizer.json python /kaggle/working/silverwing_120m_kaggle.py \
  --steps 501 --seq 2048 --batch 1 --grad-accum 4 \
  --lr 1.5e-4 --warmup 250 \
  --save-every 1 --eval-every 9999 \
  --ckpt-dir /kaggle/working/silverwing-120m/resume_test \
  --resume-from /kaggle/working/silverwing-120m/final \
  --data /kaggle/working/silverwing-120m/data/train.npy \
  --val  /kaggle/working/silverwing-120m/data/val.npy

In [ ]:
import json
md = json.load(open("/kaggle/working/silverwing-120m/resume_test/final/metadata.json"))
print("resume_step =", md["step"], "(expect 501)")
assert md["step"] == 501, "RESUME STEP MISMATCH — STOP"
print("RESUME TEST: PASS")


## 14. Evaluation (stage3a suite, same benchmarks as r2000)

In [ ]:
%%bash
python /kaggle/working/stage3a_evaluate.py \
  --ckpt /kaggle/working/silverwing-120m/final \
  --label silverwing-120m-gpu-v0.1 \
  --out /kaggle/working/silverwing-120m/evaluation_report.json

## 15. B25 capability gate (vs immutable 12M-r2000 baseline)

In [ ]:
import json
from pathlib import Path
# Attach the r2000 baseline eval as /kaggle/input/r2000-baseline/evaluation_report.json
r2000 = json.load(open("/kaggle/input/r2000-baseline/evaluation_report.json"))
rep = json.load(open("/kaggle/working/silverwing-120m/evaluation_report.json"))
hist = {"r2000_baseline": r2000, "120m_500step": rep}
json.dump(hist, open("/kaggle/working/silverwing-120m/capability_history.json","w"), indent=2)
deltas = {}
for k in ("val_loss","perplexity","math","logic","coding","knowledge"):
    a = r2000.get(k); b = rep.get(k)
    if isinstance(a,(int,float)) and isinstance(b,(int,float)):
        improved = (b < a) if k in ("val_loss","perplexity") else (b > a)
        deltas[k] = {"baseline":a, "candidate":b, "delta":b-a, "improved":improved}
print("capability deltas vs r2000:")
print(json.dumps(deltas, indent=2))
print("B25:", "PASS" if any(v["improved"] for v in deltas.values()) else "INCONCLUSIVE")


## 16. Training metrics (streaming metrics.jsonl -> final report)

In [ ]:
import json
from pathlib import Path
root = Path("/kaggle/working/silverwing-120m")
rows=[]
for line in open(root/"metrics.jsonl"):
    line=line.strip()
    if line: rows.append(json.loads(line))
json.dump(rows, open(root/"training_metrics.json","w"), indent=2)
print(f"{len(rows)} metric rows logged")
for r in rows[-6:]:
    print(r)


## 17. Sample generations

In [ ]:
%%bash
python /kaggle/working/gen_samples_120m.py \
  --ckpt /kaggle/working/silverwing-120m/final \
  --out /kaggle/working/silverwing-120m/sample_generations.txt \
  --prompts "The future of artificial intelligence" "Prove that sqrt(2) is irrational" "Explain backpropagation

## 18. Final report bundle (versioned, reproducible)

In [ ]:
import json, os, hashlib, datetime, torch
from pathlib import Path
root = Path("/kaggle/working/silverwing-120m")
md = json.load(open(root/"final/metadata.json"))
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
run_manifest = {
    "run_id": "silverwing-120m-gpu-" + md.get("model_sha256","")[:8],
    "model_version": "Silverwing-120M-V1",
    "tokenizer_version": "r2000-BPE-8192",
    "dataset_version": "silverwing-120m-mixture-v0.1",
    "hardware": gpu_name,
    "precision": md.get("amp"),
    "seed": 42,
    "ckpt_dir": str(root),
    "final_step": md["step"],
    "final_model_sha256": md.get("model_sha256"),
    "param_count": md["param_count"],
    "created": datetime.datetime.utcnow().isoformat()+"Z",
}
json.dump(run_manifest, open(root/"run_manifest.json","w"), indent=2)
env = {"python": __import__("platform").python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "bf16": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
       "fp16": bool(torch.cuda.is_available()), "gpu": gpu_name, "device": "cuda" if torch.cuda.is_available() else "cpu"}
json.dump(env, open(root/"environment_report.json","w"), indent=2)
print("run_manifest.json:"); print(json.dumps(run_manifest, indent=2))
print("environment_report.json:", json.dumps(env))
print("Final report bundle complete. Artifacts present:")
for f in ["run_manifest.json","environment_report.json","training_metrics.json","evaluation_report.json",
          "capability_history.json","dataset_manifest.json","sample_generations.txt","metrics.jsonl"]:
    p=root/f
    print(f"  {f}: {'OK' if p.exists() else 'MISSING'}")


### Decision tree & roadmap
- **CASE A** (capability improved): 500→1K→2.5K→5K→10K.  
- **CASE B** (loss↓, capability flat): inspect mixture/quality/token-budget.  
- **CASE C** (instability/NaN): lower LR, check precision, inspect grad-norm.  
- **CASE D** (OOM): batch1 → +grad-accum → grad-checkpoint → shrink seq (never shrink the model).

Instruction-tuning begins **only after** the B25 base gate — keep `Silverwing-120M-Base-v1` separate from any future `Silverwing-120M-Instruct-v1`.